In [ ]:
!pip -q install pyspark

#import statements
import os
import time
import shutil

from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("Bitcoin Heist Classification")
         .config("spark.sql.shuffle.partitions", "200")
         .config("spark.sql.adaptive.enabled", "true")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

from pyspark.sql import functions as F

#Data cleaning
DATA_PATH = "BitcoinHeistData.csv"
df_raw = spark.read.csv(DATA_PATH, header=True, inferSchema=False)

required_cols = ["year","day","length","weight","count","looped","neighbors","income","label"]

for c in required_cols:
    if c not in df_raw.columns:
        raise Exception("Missing required column: " + c)

df = df_raw.select(required_cols)

num_cols = ["year","day","length","weight","count","looped","neighbors","income"]
for c in num_cols:
    df = df.withColumn(c, F.col(c).cast("double"))

df = df.dropna(subset=num_cols + ["label"])

label_counts = df.groupBy("label").count()
valid_labels = label_counts.filter(F.col("count") >= 5).select("label")
df = df.join(valid_labels, on="label", how="inner")

df = df.sample(fraction=0.3, seed=42)

#Creating directories
WORK_DIR = "bitcoinheist_simple"
PARQUET_DIR = WORK_DIR + "/parquet_by_year"
MODELS_DIR = WORK_DIR + "/spark_models"

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

(df.repartition("year")
   .write.mode("overwrite")
   .partitionBy("year")
   .parquet(PARQUET_DIR))